Pydantic + Gemini

In [ ]:
!pip install outlines transformers torch pydantic


   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 102.3/102.3 kB 9.5 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 2.3/2.3 MB 95.1 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 67.8/67.8 kB 6.0 MB/s eta 0:00:00


In [1]:
import os
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

from Struct_Pydantic import AnnualReportExtraction # Commented out to fix ModuleNotFoundError

def process_annual_report_gemini(pdf_path: str, api_key: str, prompt: str) -> dict:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Target file not found: {pdf_path}")

    client = genai.Client(api_key=api_key)

    print(f"Uploading file to Gemini API: {pdf_path}")
    uploaded_file = client.files.upload(
        file=pdf_path,
        config={'display_name': 'Annual_Report_Target'}
    )

    generation_config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=AnnualReportExtraction,
        temperature=0.0
    )

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[uploaded_file, prompt],
            config=generation_config
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Extraction failed: {str(e)}")
        return {}
    finally:
        print(f"Deleting remote file: {uploaded_file.name}")
        client.files.delete(name=uploaded_file.name)

In [ ]:
# Execution example
api_key = ""
path_to_pdf = '1 Annual report_tagged_2025-12-11.pdf'
extraction_prompt = """
You are an expert financial auditor. Extract the required financial metrics and audit information from the attached document.
Adhere strictly to the requested JSON schema.
If a specific data point is missing, illegible, or not explicitly stated in the text, you must output null. Do not infer or guess.
"""

extraction_result = process_annual_report_gemini(path_to_pdf, api_key, extraction_prompt)
print(json.dumps(extraction_result, indent=2))

Uploading file to Gemini API: 1 Annual report_tagged_2025-12-11.pdf
Deleting remote file: files/3phekcl5yv62
{
  "parent_company": "Fidelity Qualifying Investor Funds plc",
  "financial_statement_period": {
    "value": "2024-07-31",
    "page": 1
  },
  "previous_period": {
    "value": "2023-07-31",
    "page": 1
  },
  "key_executives": [
    {
      "name": "Catherine Fitzsimons",
      "title": "Chairperson",
      "page": 203
    },
    {
      "name": "David Greco",
      "title": null,
      "page": 203
    },
    {
      "name": "Nick King",
      "title": null,
      "page": 203
    },
    {
      "name": "Bronwyn Wright",
      "title": "Independent Director",
      "page": 203
    },
    {
      "name": "Lorraine McCarthy",
      "title": null,
      "page": 203
    },
    {
      "name": "Carla Sload",
      "title": "Director",
      "page": 203
    },
    {
      "name": "Orla Buckley",
      "title": "Director",
      "page": 203
    },
    {
      "name": "Georgina Cro

In [6]:
import json
import time
from pydantic import ValidationError

def process_annual_report_with_retry(pdf_path: str, api_key: str, initial_prompt: str, max_retries: int = 2) -> dict:

    # 1. Upload initial
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Target file not found: {pdf_path}")

    client = genai.Client(api_key=api_key)

    print(f"Uploading file to Gemini API: {pdf_path}")
    uploaded_file = client.files.upload(
        file=pdf_path,
        config={'display_name': 'Annual_Report_Target'}
    )

    current_prompt = initial_prompt
    attempts = 0

    while attempts <= max_retries:
        print(f"--- Attempt {attempts + 1}/{max_retries + 1} ---")

        generation_config = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AnnualReportExtraction,
            temperature=0.0 # Crucial pour la reproductibilitÃ© lors des retries
        )

        response = client.models.generate_content(
            model='gemini-2.5-flash', # Version recommandÃ©e
            contents=[uploaded_file, current_prompt],
            config=generation_config
        )
        # --- INJECTION DE BUG POUR TEST ---
        #if attempts == 0:
        #    print("DEBUG: Simulation d'un JSON corrompu pour le test...")
        #    response = '{"ceci_n_est_pas_du_json": ...' # Force une JSONDecodeError
        # ----------------------------------
        try:
            # Tentative de validation Pydantic
            raw_json = json.loads(response.text)
            validated_data = AnnualReportExtraction(**raw_json)

            print("Extraction successful and validated.")
            return raw_json

        except (ValidationError, json.JSONDecodeError, Exception) as e:
            attempts += 1
            if attempts > max_retries:
                print("Nombre maximal de tentatives atteint.")
                return raw_json if 'raw_json' in locals() else {}

            # Gestion de l'erreur et prÃ©paration du retry
            error_message = str(e)
            print(f"Ã‰chec de validation ou erreur API : {error_message}")

            current_prompt = (
                f"{initial_prompt}\n\n"
                f"CORRECTION REQUISE : Ton prÃ©cÃ©dent rÃ©sultat a gÃ©nÃ©rÃ© l'erreur suivante :\n"
                f"{error_message}\n\n"
                "Analyse Ã  nouveau le document et corrige les donnÃ©es en respectant strictement le schÃ©ma."
            )

    return {}

In [ ]:
# Execution example
api_key = ""
path_to_pdf = '1 Annual report_tagged_2025-12-11.pdf'
extraction_prompt = """
You are an expert financial auditor. Extract the required financial metrics and audit information from the attached document.
Adhere strictly to the requested JSON schema.
If a specific data point is missing, illegible, or not explicitly stated in the text, you must output null. Do not infer or guess.
"""

extraction_result = process_annual_report_with_retry(path_to_pdf, api_key, extraction_prompt)
print(json.dumps(extraction_result, indent=2))

Uploading file to Gemini API: 1 Annual report_tagged_2025-12-11.pdf
--- Attempt 1/3 ---
DEBUG: Simulation d'un JSON corrompu pour le test...
Ã‰chec de validation ou erreur API : 'str' object has no attribute 'text'
Respect du Rate Limit : Pause de 5 secondes avant le prochain essai...
--- Attempt 2/3 ---
Extraction successful and validated.
{
  "parent_company": "Fidelity Qualifying Investor Funds plc",
  "financial_statement_period": {
    "value": "2024-07-31",
    "page": 1
  },
  "previous_period": {
    "value": "2023-07-31",
    "page": 1
  },
  "key_executives": [
    {
      "name": "Carla Sload",
      "title": "Secretary",
      "page": 1
    },
    {
      "name": "Georgina Cromwell",
      "title": "Secretary",
      "page": 1
    },
    {
      "name": "Ms. Catherine Fitzsimons",
      "title": "Chairperson",
      "page": 203
    },
    {
      "name": "Ms. Bronwyn Wright",
      "title": null,
      "page": 203
    },
    {
      "name": "Ms. Orla Buckley",
      "title"

Outlines

In [9]:
import json
import torch
import outlines
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from Struct_Outlines import AnnualReportExtraction

def extract_with_powerful_model(report_text, instructions, model_id="microsoft/Phi-3-mini-4k-instruct"):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    try:
        hf_model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
        )
        hf_tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = outlines.models.transformers(hf_model, hf_tokenizer)

        # SOLUTION : On passe par le JSON Schema pur pour Ã©viter l'erreur 'type'
        schema_json = json.dumps(AnnualReportExtraction.model_json_schema())
        generator = outlines.generate.json(model, schema_json)

        full_prompt = f"<|user|>\n{instructions}\n\nContexte :\n{report_text}<|end|>\n<|assistant|>"

        # Le gÃ©nÃ©rateur renvoie un dictionnaire
        result_dict = generator(full_prompt, max_tokens=1536)

        # On valide avec Pydantic pour la forme
        return AnnualReportExtraction.model_validate(result_dict).model_dump()

    except Exception as e:
        return {"error": f"Extraction failed: {str(e)}"}

# --- Bloc d'exÃ©cution ---
instructions = "Extract the target financial metrics from the following document context."
path_to_pdf = '1 Annual report_tagged_2025-12-11.pdf'
result = extract_with_powerful_model(path_to_pdf, instructions)
print(json.dumps(result, indent=4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


{
    "error": "Extraction failed: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`"
}
